# Week 2: Mini Semantic Search

## Project Overview
Build a simple semantic search engine that retrieves FAQs based on meaning using Python, Pandas, Sentence Transformers, and ChromaDB.

In [4]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Python executable: /usr/bin/python3


## 1. Install Required Packages

We will use the packages listed in the existing requirements.txt file from Week 1, adding the new dependencies required for semantic search.

In [5]:
%pip install pandas chromadb sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 98.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 102.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 91.6 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      Su

In [6]:
import pandas as pd
import chromadb
import sentence_transformers

print("Pandas version:", pd.__version__)
print("ChromaDB version:", chromadb.__version__)
print("Sentence Transformers installed successfully!")

Pandas version: 2.2.3
ChromaDB version: 1.5.9
Sentence Transformers installed successfully!


In [7]:
from google.colab import files

uploaded = files.upload()

Saving FAQs.csv to FAQs.csv


In [9]:
import pandas as pd

df = pd.read_csv("FAQs.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (10, 2)
Columns: ['question', 'answer']


,question,answer
0,What is Python?,Python is a high-level programming language.
1,How do I install Python?,Download Python from python.org and follow the...
2,What is Git?,Git is a version control system for tracking c...
3,What is GitHub?,GitHub is an online platform for hosting and c...
4,What is machine learning?,Machine learning enables computers to learn pa...


In [10]:
df = df.dropna(subset=["question", "answer"]).copy()

df["question"] = df["question"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
].reset_index(drop=True)

print("FAQs after cleaning:", len(df))
df.head()

FAQs after cleaning: 10


,question,answer
0,What is Python?,Python is a high-level programming language.
1,How do I install Python?,Download Python from python.org and follow the...
2,What is Git?,Git is a version control system for tracking c...
3,What is GitHub?,GitHub is an online platform for hosting and c...
4,What is machine learning?,Machine learning enables computers to learn pa...


In [11]:
df["text"] = (
    "Question: " + df["question"]
    + "\nAnswer: " + df["answer"]
)

df[["question", "answer", "text"]].head()

,question,answer,text
0,What is Python?,Python is a high-level programming language.,Question: What is Python?\nAnswer: Python is a...
1,How do I install Python?,Download Python from python.org and follow the...,Question: How do I install Python?\nAnswer: Do...
2,What is Git?,Git is a version control system for tracking c...,Question: What is Git?\nAnswer: Git is a versi...
3,What is GitHub?,GitHub is an online platform for hosting and c...,Question: What is GitHub?\nAnswer: GitHub is a...
4,What is machine learning?,Machine learning enables computers to learn pa...,Question: What is machine learning?\nAnswer: M...


In [12]:
chunk_size = 3
documents = []

for start in range(0, len(df), chunk_size):
    chunk = df["text"].iloc[start:start + chunk_size]
    documents.append("\n\n".join(chunk))

print("Total FAQs:", len(df))
print("Total chunks:", len(documents))

print("\nFirst chunk:\n")
print(documents[0])

Total FAQs: 10
Total chunks: 4

First chunk:

Question: What is Python?
Answer: Python is a high-level programming language.

Question: How do I install Python?
Answer: Download Python from python.org and follow the installation instructions.

Question: What is Git?
Answer: Git is a version control system for tracking changes in code.


In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [14]:
faq_documents = df["text"].tolist()

embeddings = model.encode(
    faq_documents,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Number of FAQ documents:", len(faq_documents))
print("Embedding shape:", embeddings.shape)
print("First embedding:")
print(embeddings[0][:10])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of FAQ documents: 10
Embedding shape: (10, 384)
First embedding:
[-0.03043334  0.02783153 -0.01722443  0.00098189 -0.02475169 -0.13222194
 -0.02178487  0.07724977 -0.10069434 -0.01575132]


In [15]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="faq_collection",
    metadata={"hnsw:space": "cosine"}
)

collection.upsert(
    ids=[f"faq_{i}" for i in range(len(faq_documents))],
    documents=faq_documents,
    embeddings=embeddings.tolist()
)

print("Stored FAQs in ChromaDB:", collection.count())

Stored FAQs in ChromaDB: 10


In [16]:
query = "How can I track changes in my code?"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=min(3, len(faq_documents))
)

for rank, document in enumerate(results["documents"][0], start=1):
    distance = results["distances"][0][rank - 1]

    print(f"\n{rank}. {document}")
    print(f"Cosine distance: {distance:.4f}")


1. Question: What is Git?
Answer: Git is a version control system for tracking changes in code.
Cosine distance: 0.5528

2. Question: What is GitHub?
Answer: GitHub is an online platform for hosting and collaborating on Git repositories.
Cosine distance: 0.8707

3. Question: What is an API?
Answer: An API allows different software applications to communicate.
Cosine distance: 0.8773


In [ ]:
while True:
    query = input("\nEnter your question (or type 'exit'): ").strip()

    if query.lower() == "exit":
        print("Search ended. Goodbye!")
        break

    if not query:
        print("Please enter a question.")
        continue

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(3, len(faq_documents))
    )

    print("\nTop 3 matching FAQs:")

    for rank, document in enumerate(
        results["documents"][0], start=1
    ):
        distance = results["distances"][0][rank - 1]

        print(f"\n{rank}. {document}")
        print(f"Cosine distance: {distance:.4f}")


Enter your question (or type 'exit'): What is an API?

Top 3 matching FAQs:

1. Question: What is an API?
Answer: An API allows different software applications to communicate.
Cosine distance: 0.1679

2. Question: What is Python?
Answer: Python is a high-level programming language.
Cosine distance: 0.6259

3. Question: What is artificial intelligence?
Answer: Artificial intelligence enables machines to perform tasks that typically require human intelligence.
Cosine distance: 0.7643

Enter your question (or type 'exit'): What is python?

Top 3 matching FAQs:

1. Question: What is Python?
Answer: Python is a high-level programming language.
Cosine distance: 0.1530

2. Question: How do I install Python?
Answer: Download Python from python.org and follow the installation instructions.
Cosine distance: 0.4525

3. Question: What is Pandas?
Answer: Pandas is a Python library for data manipulation and analysis.
Cosine distance: 0.4796
